# 02 — Data Cleaning, EDA & Modeling Preparation  
## Forecasting and Identifying Global Export Opportunities for Algerian Exporters  
**ENSIA — Machine Learning Project — Spring 2025/2026**

---

## Notebook Purpose

This notebook starts from the integrated master dataset produced in Notebook 01:

`../data/master_df.parquet`

The goal of this notebook is to continue **Step 2: Data Preparation & Integration** and begin the **Exploratory Data Analysis (EDA)** required by the project.

It focuses on:

- checking the quality of the integrated dataset,
- identifying missing values and inconsistencies,
- preparing a clear imputation strategy,
- preparing normalization and scaling for future machine learning models,
- analyzing Algerian export flows,
- studying product and country demand patterns,
- visualizing export opportunities using tables, charts, and heatmaps,
- saving cleaned outputs for later modeling and dashboard notebooks.

---

## Notebook Sections

This notebook is organized as follows:

1. Load and audit the master dataset.
2. Create a data source audit table.
3. Run data quality checks: duplicates, data types, and inconsistencies.
4. Analyze missing values and define an imputation plan.
5. Inspect the target variable `label_opportunity`.
6. Prepare the normalization and scaling plan.
7. EDA: Algeria's export flows over time.
8. EDA: Top export partners.
9. EDA: Top exported products.
10. EDA: Global demand analysis.
11. EDA: Sector and product demand analysis.
12. EDA: Opportunity countries and products.
13. EDA: Country-product heatmaps.
14. Prepare modeling-ready datasets.
15. Save EDA outputs for dashboard preparation.

---

## Relation to Project Requirements

This notebook directly addresses the project requirements related to:

- cleaning and harmonizing multi-source datasets,
- handling missing values and inconsistencies,
- preparing normalization and scaling,
- validating engineered features such as export growth, global demand, market penetration, trade balance, and diversification indicators,
- producing exploratory visualizations of trade flows, sector demand, and country-product opportunities.

The outputs of this notebook will be used in the next stages: clustering, classification, forecasting, and dashboard development.

## 0. Imports and Configuration

In this section, we import the libraries needed for data cleaning, exploratory data analysis, and visualization.

We also define the main project paths. Since this notebook is inside the `notebooks` folder, the code automatically moves one level up to find the `data` folder and the master dataset created in Notebook 01.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:,.2f}".format)
plt.rcParams["figure.dpi"] = 110

ROOT       = Path("..") if Path.cwd().name == "notebooks" else Path(".")
DATA_DIR   = ROOT / "data"
MASTER_PATH = DATA_DIR / "master_df.parquet"

assert MASTER_PATH.exists(), f"File not found: {MASTER_PATH} — run notebook 01 first."
print("Setup complete.")
print(f"Master file: {MASTER_PATH}")

: 

---
## Section 1 — Load Master Dataset

We load only the columns needed for EDA to avoid loading the full 14.15M × 50 dataframe into memory all at once.
The dataset grain is: one row per **(year, partner country, HS6 product)** — including pairs where Algeria currently exports zero.

In [ ]:
import gc

# Clear memory if rerun
if "df" in globals():
    del df
gc.collect()

# Very light version: no country_name, no description_short
# We will add names later only in small summary tables.
use_cols = [
    "t", "j", "k", "iso3", "split",
    "label_opportunity", "is_new_entry",
    "alg_export_v", "partner_import_v", "world_import_v",
    "market_penetration", "global_demand_log", "global_demand_rank",
    "world_demand_growth", "rca",
    "gdp_usd", "population", "gdp_per_capita",
    "hhi_export", "diversification_index"
]

print("Loading lightweight EDA columns...")

df = pd.read_parquet(
    MASTER_PATH,
    columns=use_cols,
    engine="fastparquet"
)

# Reduce memory after loading
for col in ["k", "iso3", "split"]:
    df[col] = df[col].astype("category")

for col in ["t", "j", "label_opportunity", "is_new_entry"]:
    df[col] = pd.to_numeric(df[col], downcast="integer")

float_cols = df.select_dtypes(include="float").columns
for col in float_cols:
    df[col] = pd.to_numeric(df[col], downcast="float")

print("Dataset loaded successfully.")
print(f"Shape: {df.shape}")
print(f"Years: {[int(y) for y in sorted(df['t'].unique())]}")
print(f"Partners: {df['j'].nunique()}")
print(f"Products: {df['k'].nunique()}")

df.head(3)

The lightweight EDA dataframe was loaded successfully.

It contains the full 14,153,904 rows, but only 20 selected columns. This keeps the complete year-partner-product panel while reducing memory usage.

Text-heavy columns such as country names and product descriptions were excluded from the full dataframe to avoid memory errors. They can be added later only to small aggregated summary tables when needed.

In [ ]:
df.info(memory_usage="deep")

The memory audit confirms that the lightweight dataframe is much more efficient.

After selecting only the necessary columns and converting text identifiers to `category`, the memory usage decreased to about 1.1 GB. This makes the dataset easier to analyze in the notebook without crashing the environment.

The data types are also appropriate:
- `t`, `j`, `label_opportunity`, and `is_new_entry` are stored as compact integer types,
- `k`, `iso3`, and `split` are stored as categorical variables,
- most engineered trade and economic variables are numeric.

This confirms that the dataset is loaded in a memory-efficient format and is ready for data quality checks and EDA.

---
## Section 2 — Data Source Audit

This table summarizes all data sources used in the project, their role, and their current integration status.

In [ ]:
data_sources = pd.DataFrame([
    {"Source": "BACI HS12",        "Used": "Yes",              "Role": "Bilateral trade flows by product, partner, year",             "Status": "Integrated"},
    {"Source": "BACI country codes","Used": "Yes",              "Role": "Numeric code → ISO3 / country name mapping",                 "Status": "Integrated"},
    {"Source": "BACI product codes","Used": "Yes",              "Role": "HS6 product descriptions",                                  "Status": "Integrated"},
    {"Source": "World Bank API",    "Used": "Yes",              "Role": "GDP, population, trade openness, GDP per capita",            "Status": "Integrated"},
    {"Source": "UNCTAD",            "Used": "Yes",              "Role": "Algeria export concentration and diversification index",     "Status": "Integrated"},
    {"Source": "CEPII GeoDist", "Used": "Yes", "Role": "Distance, common language, border, colonial links", "Status": "Integrated — 92.5% coverage"},
    {"Source": "CACI / Ministry",   "Used": "No",              "Role": "Official Algerian export support data (not publicly available)","Status": "Not available"},
])

data_sources

The data source audit confirms that the project integrates several public international datasets.

BACI is the main trade data source. World Bank adds macroeconomic indicators, CEPII GeoDist adds geographic and cultural proximity variables, and UNCTAD adds Algeria-level diversification indicators.

CACI or Ministry data was not used because it was not publicly available for this project. Therefore, the project remains based on publicly accessible and reproducible data sources.

## Section 3 — Data Quality Checks

Before starting EDA, we first inspect the basic structure of the dataset: number of rows, years, partners, products, and split distribution.

After this overview, we will run additional checks for duplicates, infinite values, missing values, and inconsistent feature ranges.

In [ ]:
print("=== Basic statistics ===")
print(f"Shape:    {df.shape}")
print(f"Years:    {[int(y) for y in sorted(df['t'].unique())]}")
print(f"Partners: {df['j'].nunique()}")
print(f"Products: {df['k'].nunique()}")
print()
print("=== Split distribution ===")
print(df['split'].value_counts())

The dataset overview confirms that the loaded dataframe contains 14,153,904 rows and 20 selected columns.

The data covers the full period from 2012 to 2023, with 227 partner countries and 5,196 HS6 products.

The temporal split is consistent with the project design:
- training set: 2012–2019,
- validation set: 2020–2021,
- test set: 2022–2023.

This confirms that the full year-partner-product panel created in Notebook 01 was preserved after loading the lightweight EDA dataset.

In [ ]:
# Check for duplicate rows on the grain key
duplicates = df.duplicated(subset=["t", "j", "k"]).sum()
print(f"Duplicate (t, j, k) rows: {duplicates}")
if duplicates == 0:
    print("  ✓ No duplicates — dataset grain is correct.")
else:
    print("  ✗ Duplicates found — investigate before modelling.")

In [ ]:
# Check for infinite values in numeric columns
num_cols = df.select_dtypes(include=["number"]).columns
inf_check = np.isinf(df[num_cols]).sum().sort_values(ascending=False)
inf_found = inf_check[inf_check > 0]

if len(inf_found) == 0:
    print("✓ No infinite values found in any numeric column.")
else:
    print("✗ Infinite values detected:")
    print(inf_found)

In [ ]:
# Verify numeric column value ranges
range_checks = {
    "market_penetration": (0, 1),
    "global_demand_rank": (0, 1),
    "label_opportunity":  (0, 1),
}
for col, (lo, hi) in range_checks.items():
    if col in df.columns:
        valid = df[col].dropna().between(lo, hi).all()
        print(f"  {'✓' if valid else '✗'}  {col} in [{lo}, {hi}]")

In [ ]:
valid_splits = set(df["split"].unique()).issubset({"train", "val", "test"})
print(f"{'✓' if valid_splits else '✗'} split values are valid")

The structural quality checks passed successfully.

There are no duplicate `(year, partner, product)` rows, which confirms that the dataset grain is correct. No infinite values were found in numeric columns, meaning problematic divisions were handled correctly during feature engineering.

The bounded variables are also within their expected ranges:
- `market_penetration` is between 0 and 1,
- `global_demand_rank` is between 0 and 1,
- `label_opportunity` is binary.

These checks confirm that the dataset is structurally consistent and ready for missing value analysis.

---

## Section 4 — Missing Value Analysis and Imputation Plan

After checking the structural quality of the dataset, the next step is to analyze missing values.

Missing values are normal in this project because the dataset is built from several different sources. Each source has its own coverage. For example, BACI covers international trade flows, World Bank covers macroeconomic indicators, and UNCTAD provides diversification indicators. Because these sources do not always cover exactly the same countries, products, and years, some variables contain missing values.

The goal of this section is not to fill missing values immediately, but to understand where they appear and decide how they should be handled later in the modeling pipeline.

In this notebook, we analyze missing values for the selected lightweight EDA dataframe. The main causes of missing values are:

- **World Bank indicators**: some small territories or special country codes in BACI do not have complete macroeconomic data in the World Bank database.
- **RCA**: the Revealed Comparative Advantage ratio can be missing when Algeria has no export activity for a product-year combination, so the ratio is not defined.
- **UNCTAD diversification index**: in this version of the dataset, `diversification_index` is fully missing, so it should not be used as a modeling feature unless the source file is corrected.

It is important not to fill missing values before splitting the data for modeling.  
Any imputation used for machine learning must be fitted only on the training set, then applied to the validation and test sets. This prevents data leakage from future periods into the training process.

In [ ]:
missing = (
    df.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .reset_index()
)
missing.columns = ["column", "missing_percent"]
missing[missing["missing_percent"] > 0]

The missing value table shows which columns contain missing values and how large the problem is for each variable.

The most important result is that `diversification_index` is 100% missing. This means the column does not contain usable information in the current dataset and should be removed from the modeling feature set unless the UNCTAD source file is corrected.

The `rca` variable has a high missing rate. This is expected because RCA cannot always be computed for product-year combinations where Algeria has no export activity. Since the full opportunity grid intentionally includes many zero-export rows, missing RCA values are normal.

World Bank variables such as `gdp_usd`, `gdp_per_capita`, and `population` also contain missing values. This is acceptable because some partner territories in BACI do not have full macroeconomic coverage in World Bank data.

In [ ]:
# Visualize missing values
miss_plot = missing[missing["missing_percent"] > 0]

if len(miss_plot) > 0:
    fig, ax = plt.subplots(figsize=(9, max(3, len(miss_plot) * 0.45)))
    ax.barh(miss_plot["column"][::-1], miss_plot["missing_percent"][::-1], color="steelblue")
    ax.set_xlabel("Missing (%)")
    ax.set_title("Missing Values by Column")
    ax.axvline(x=10, color="orange", linestyle="--", linewidth=1, label="10% threshold")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No missing values found.")

The missing value plot confirms the same pattern visually.

Only a small number of selected variables have missing values. The strongest issue is `diversification_index`, which is fully missing. RCA also has many missing values, while World Bank variables have moderate missingness.

This plot helps prioritize the cleaning strategy: fully missing variables should be dropped, while partially missing numeric variables can be handled using imputation during the modeling stage.

In [ ]:
# Imputation plan table
numeric_features = [
    "alg_export_growth", "world_demand_growth", "global_demand_log",
    "global_demand_rank", "market_penetration", "trade_balance_bilateral",
    "trade_coverage_ratio", "n_export_destinations", "rca",
    "gdp_usd", "population", "trade_pct_gdp", "gdp_per_capita", "market_size_usd",
    "hhi_export", "diversification_index"
]

available_features = [c for c in numeric_features if c in df.columns]

rows = []

for col in available_features:
    missing_pct = round(df[col].isna().mean() * 100, 2)

    if missing_pct == 100:
        strategy = "Drop from modeling because the column is fully missing"
    elif missing_pct > 0:
        strategy = "Median imputation fitted on train split only"
    else:
        strategy = "No imputation needed"

    rows.append({
        "feature": col,
        "missing_percent": missing_pct,
        "planned_strategy": strategy
    })

imputation_plan = pd.DataFrame(rows)

imputation_plan

The imputation plan separates variables into three groups.

Variables with 0% missing values do not require imputation. These include several important demand and penetration indicators such as `world_demand_growth`, `global_demand_log`, `global_demand_rank`, `market_penetration`, and `hhi_export`.

Variables with partial missing values, such as `rca`, `gdp_usd`, `population`, and `gdp_per_capita`, will be imputed later using the median value calculated only from the training set. This avoids data leakage and keeps the validation and test sets independent.

The `diversification_index` column is fully missing, so it should be dropped from the modeling feature set. Keeping a fully missing variable would not help the model and could create problems during preprocessing.

The missing value analysis shows that the dataset is mostly usable for EDA and modeling preparation.

Most key trade-demand features are complete. The main cleaning decisions are:

- drop `diversification_index` from modeling because it is 100% missing,
- impute partially missing macroeconomic variables using the training-set median,
- impute or flag missing `rca` values during modeling,
- avoid fitting any imputation strategy on validation or test data.

This approach keeps the preprocessing pipeline clean, reproducible, and free from data leakage.

---
## Section 5 — Target Label Inspection

We verify the distribution of the binary opportunity label. A target ratio of 15–25% positives gives a good balance for classification without requiring heavy class-weight adjustments.

In [ ]:
label_dist = (
    df["label_opportunity"]
    .value_counts(normalize=True)
    .mul(100)
    .rename_axis("label")
    .reset_index(name="percent")
)
label_dist["label"] = label_dist["label"].map({0: "No opportunity", 1: "Opportunity"})

print(label_dist.to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(label_dist["label"], label_dist["percent"], color=["#c0392b", "#27ae60"])
ax.set_ylabel("Share (%)")
ax.set_title("Distribution of the Export Opportunity Label")
for i, v in enumerate(label_dist["percent"]):
    ax.text(i, v + 0.5, f"{v:.1f}%", ha="center", fontsize=11)
plt.tight_layout()
plt.show()

The target label distribution is acceptable for classification.

Around 21.25% of the observations are labeled as export opportunities, while 78.75% are labeled as non-opportunities. This means the dataset is moderately imbalanced, but not severely imbalanced.

The opportunity class is large enough for the model to learn meaningful patterns. Later, during model evaluation, metrics such as precision, recall, and F1-score will be more informative than accuracy alone because the classes are not perfectly balanced.

This distribution also confirms that the percentile-based labeling strategy produced a reasonable number of positive opportunity cases.

## Section 6 — Normalization and Scaling Plan

Scaling is required for some machine learning methods because the features are measured on very different scales. For example, GDP can be in billions of dollars, while market penetration is between 0 and 1.

Scaling is especially important for:

- **Clustering models** such as k-means and hierarchical clustering, because they rely on distances between observations.
- **Scale-sensitive classifiers** such as logistic regression, SVM, and neural networks.
- **Gradient-based models**, where large-scale variables may dominate the optimization process.

Scaling is usually not required for tree-based models such as Random Forest and XGBoost, because decision trees split variables based on thresholds and are less sensitive to feature scale.

In this notebook, we demonstrate the correct preprocessing logic. Missing values are imputed first, then features are scaled. The imputer and scaler are fitted only on the training split, then applied to validation and test sets. This avoids data leakage.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# ── Scaling feature list ───────────────────────────────────────────────────
# Numeric features used by scale-sensitive models (logistic regression,
# SVM, k-means, neural nets). Tree-based models (Random Forest, XGBoost)
# do not need scaling — they read raw columns from the full split parquets
# saved in Section 14.
#
# Note: diversification_index is intentionally excluded — it was found to be
# 100% missing in Section 4 and must not be used as a modelling feature.

SCALE_FEATURES = [
    # BACI trade features
    "alg_export_growth",
    "world_demand_growth",
    "global_demand_log",
    "global_demand_rank",
    "market_penetration",
    "trade_balance_bilateral",
    "trade_coverage_ratio",
    "n_export_destinations",
    "rca",
    # Lag features (already on log1p scale from Notebook 01)
    "alg_export_v_lag1",       "alg_export_v_lag2",
    "market_penetration_lag1", "market_penetration_lag2",
    "world_import_v_lag1",     "world_import_v_lag2",
    "rca_lag1",                "rca_lag2",
    # GeoDist
    "dist_km", "distw_km", "contig",
    "comlang_off", "comlang_ethno", "colony", "comcol", "smctry",
    # World Bank
    "gdp_usd", "population", "trade_pct_gdp", "gdp_per_capita", "market_size_usd",
    # UNCTAD
    "hhi_export",
    # diversification_index excluded — 100% missing (see Section 4)
]

# Preprocessing pipeline — DEFINED here, FITTED in Section 14 on the full
# train split. Deferring the fit avoids any risk of val/test leakage.
preprocess = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])

print(f"Scale features defined: {len(SCALE_FEATURES)} columns")
print(SCALE_FEATURES)
print()
print("Pipeline defined. Fitting is deferred to Section 14 where the full")
print("50-column train split is available — correct, no data leakage.")


The scaling feature list and preprocessing pipeline were defined successfully.

`SCALE_FEATURES` contains all numeric columns from the full master dataset that benefit from imputation and scaling: BACI trade features, lag features, GeoDist variables, World Bank indicators, and the UNCTAD HHI. `diversification_index` is excluded because it is 100% missing.

The pipeline is only **defined** here — it is not fitted. Fitting happens in Section 14 once the full 50-column splits are loaded, ensuring the imputation medians are computed only from training-period data and that the validation and test sets remain completely unseen.

Scale-sensitive models (logistic regression, SVM, k-means, neural networks) will use the scaled arrays. Tree-based models (Random Forest, XGBoost) can read directly from the full-column parquet splits without needing scaling.

---
## Section 7 — EDA: Algeria's Export Value Over Time

We start with a macro view: how did Algeria's total recorded export value in BACI evolve from 2012 to 2023?

BACI values are in thousand USD, so we multiply by 1,000 for interpretability.

In [ ]:
exports_by_year = (
    df.groupby("t", as_index=False)["alg_export_v"]
    .sum()
)
exports_by_year["alg_export_usd"] = exports_by_year["alg_export_v"] * 1_000

print(exports_by_year[["t", "alg_export_usd"]].to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(exports_by_year["t"], exports_by_year["alg_export_usd"] / 1e9,
        marker="o", linewidth=2, color="#2980b9")
ax.fill_between(exports_by_year["t"], exports_by_year["alg_export_usd"] / 1e9,
                alpha=0.15, color="#2980b9")
ax.set_title("Algeria Total Export Value Over Time", fontsize=13)
ax.set_xlabel("Year")
ax.set_ylabel("Export Value (billion USD)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.0f}B"))
ax.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

The export value trend shows a clear decline after 2014, followed by a recovery after 2020.

Algeria's total export value was highest during 2012–2014, then decreased sharply between 2015 and 2016. This reflects the volatility of Algeria's export structure, which is strongly influenced by global energy markets.

The lowest point appears in 2020, which can be linked to the global trade slowdown during the COVID-19 period. After that, exports increased again in 2021 and reached a strong recovery in 2022, before decreasing slightly in 2023.

This macro-level view is useful because it shows that Algerian exports are volatile over time. For this reason, the project should not only analyze total export value, but also look deeper into products, partners, demand, and opportunity markets.

---
## Section 8 — EDA: Top Algerian Export Partners

Which countries receive the most Algerian exports (by value over the full 2012–2023 period)?

This reveals Algeria's current geographic concentration.

In [ ]:
# Build a small partner lookup table from the master parquet
partner_lookup = (
    pd.read_parquet(
        MASTER_PATH,
        columns=["j", "country_name"],
        engine="fastparquet"
    )
    .dropna()
    .drop_duplicates()
)

# Aggregate Algeria's exports by partner code
top_partners = (
    df.groupby("j", as_index=False)["alg_export_v"]
    .sum()
    .sort_values("alg_export_v", ascending=False)
    .head(15)
)

# Attach readable country names
top_partners = top_partners.merge(partner_lookup, on="j", how="left")
# Fix encoding issue for Türkiye
top_partners["country_name"] = (
    top_partners["country_name"]
    .astype(str)
    .str.replace("TÃ¼rkiye", "Türkiye", regex=False)
    .str.replace("TÃƒÂ¼rkiye", "Türkiye", regex=False)
)

# Convert to USD
top_partners["alg_export_usd"] = top_partners["alg_export_v"] * 1_000

# Sort again just to be safe
top_partners = top_partners.sort_values("alg_export_usd", ascending=False)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(
    top_partners["country_name"][::-1],
    top_partners["alg_export_usd"][::-1] / 1e9,
    color="#2980b9"
)

ax.set_title("Top 15 Algerian Export Partners (2012–2023)", fontsize=13)
ax.set_xlabel("Total Export Value (billion USD)")
ax.set_ylabel("Partner Country")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.0f}B"))
ax.grid(True, axis="x", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

top_partners[["country_name", "alg_export_usd"]]

The top export partner analysis shows that Algeria’s exports are geographically concentrated in a limited number of destination markets.

Italy, Spain, and France are the largest export partners over the 2012–2023 period. This reflects Algeria’s strong trade links with nearby European markets. The USA, United Kingdom, and Netherlands also appear among the main destinations.

This concentration suggests that Algeria’s export performance depends heavily on a relatively small group of partner countries. From an export diversification perspective, relying on a few major destinations can increase vulnerability to external demand shocks.

A minor encoding issue appeared for Türkiye in the raw country name. This was corrected only for display purposes and does not affect the numerical analysis.

---
## Section 9 — EDA: Top Exported Products

Which HS6 products account for most of Algeria's export value?
This directly shows the current export basket and its level of diversification.

In [ ]:
# Build a small product lookup table from the master parquet
# We do this because the lightweight df does not contain description_short
product_lookup = (
    pd.read_parquet(
        MASTER_PATH,
        columns=["k", "description_short"],
        engine="fastparquet"
    )
    .dropna()
    .drop_duplicates()
)

# Aggregate Algeria's exports by HS6 product code
top_products = (
    df.groupby("k", as_index=False, observed=True)["alg_export_v"]
    .sum()
    .sort_values("alg_export_v", ascending=False)
    .head(15)
)

# Attach readable product descriptions
top_products = top_products.merge(product_lookup, on="k", how="left")

# Convert BACI values from thousand USD to USD
top_products["alg_export_usd"] = top_products["alg_export_v"] * 1_000

# Sort again by USD value
top_products = top_products.sort_values("alg_export_usd", ascending=False)

# Plot top exported products
fig, ax = plt.subplots(figsize=(11, 6))

ax.barh(
    top_products["description_short"][::-1],
    top_products["alg_export_usd"][::-1] / 1e9,
    color="#16a085"
)

ax.set_title("Top 15 Algerian Exported Products (2012–2023)", fontsize=13)
ax.set_xlabel("Total Export Value (billion USD)")
ax.set_ylabel("HS6 Product")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.0f}B"))
ax.grid(True, axis="x", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

top_products[["k", "description_short", "alg_export_usd"]]

The top exported product analysis shows that Algeria’s export basket is highly concentrated in hydrocarbons.

The largest exported products are petroleum oils, crude petroleum oils, and petroleum gases. These products dominate the total export value over the 2012–2023 period. This confirms the economic context of the project: Algeria’s exports are still strongly dependent on the energy sector.

Some non-hydrocarbon products also appear, such as fertilizers, ammonia, sugar, cement clinker, and iron or steel products, but their export values are much smaller compared to petroleum-related products.

This result highlights the importance of export diversification. Since the current export basket is concentrated in a small number of products, the machine learning system should help identify new high-demand products and promising markets where Algeria could expand exports.

Several HS6 codes belong to the same broad petroleum or gas sector, which explains why similar product descriptions appear multiple times in the ranking.

---
## Section 10 — EDA: Global Demand Analysis

Which HS6 products are most demanded globally (measured by world import value)?

These are the products where there is the most potential market to capture, independent of Algeria's current position.

Important: `world_import_v` is attached to every partner-product row in the full opportunity grid. Therefore, before aggregating global demand, we keep only one row per `(year, product)` to avoid counting the same global demand value multiple times.

In [ ]:
# Build a small product lookup table from the master parquet
product_lookup = (
    pd.read_parquet(
        MASTER_PATH,
        columns=["k", "description_short"],
        engine="fastparquet"
    )
    .dropna()
    .drop_duplicates()
)

# world_import_v is repeated for every partner in the full grid.
# To avoid double-counting, keep only one row per (year, product).
world_product_year = (
    df[["t", "k", "world_import_v"]]
    .drop_duplicates(subset=["t", "k"])
)

# Aggregate true global import demand by HS6 product
global_demand_products = (
    world_product_year
    .groupby("k", as_index=False, observed=True)["world_import_v"]
    .sum()
    .sort_values("world_import_v", ascending=False)
    .head(20)
)

# Attach readable product descriptions
global_demand_products = global_demand_products.merge(
    product_lookup,
    on="k",
    how="left"
)

# Convert BACI values from thousand USD to USD
global_demand_products["world_import_usd"] = (
    global_demand_products["world_import_v"] * 1_000
)

# Sort again by USD value
global_demand_products = global_demand_products.sort_values(
    "world_import_usd",
    ascending=False
)

# Plot top global demand products
fig, ax = plt.subplots(figsize=(11, 7))

ax.barh(
    global_demand_products["description_short"][::-1],
    global_demand_products["world_import_usd"][::-1] / 1e12,
    color="#8e44ad"
)

ax.set_title("Top 20 Products by Global Import Demand (2012–2023)", fontsize=13)
ax.set_xlabel("World Import Value (trillion USD)")
ax.set_ylabel("HS6 Product")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.1f}T"))
ax.grid(True, axis="x", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

global_demand_products[["k", "description_short", "world_import_usd"]]

The corrected global demand analysis shows the HS6 products with the highest total world import demand over 2012–2023.

The largest global demand products include petroleum oils, crude petroleum oils, gold, medicaments, mobile phones, vehicles, electronic circuits, petroleum gases, computers, and communication equipment.

This result is important because it separates global market demand from Algeria's current export performance. Some high-demand products, such as petroleum oils and petroleum gases, overlap with Algeria's current export strengths. However, other high-demand products, such as medicaments, electronics, vehicles, and communication equipment, may represent areas where Algeria has lower current participation but where global demand is very large.

Before aggregating, we kept only one row per `(year, product)` because `world_import_v` is repeated across partner rows in the full opportunity grid. This avoided double-counting and produced realistic global demand values.

---
## Section 11 — EDA: Sector Demand Analysis

We map each HS6 product to its HS section (broad economic sector) using the first two digits of the product code. This gives a **sector-level view** of:
- Where Algeria currently exports (actual export value by sector)
- Where global demand is concentrated (world import value by sector)
- How many opportunity rows exist per sector

In [ ]:
# Build HS2 sector mapping
df["hs2"] = df["k"].astype(str).str[:2].astype("int16")

def hs_section(hs2):
    if   1  <= hs2 <= 5:   return "Animal products"
    elif 6  <= hs2 <= 14:  return "Vegetable products"
    elif 15 <= hs2 <= 15:  return "Animal/vegetable fats"
    elif 16 <= hs2 <= 24:  return "Food, beverages, tobacco"
    elif 25 <= hs2 <= 27:  return "Mineral products"
    elif 28 <= hs2 <= 38:  return "Chemicals"
    elif 39 <= hs2 <= 40:  return "Plastics and rubber"
    elif 41 <= hs2 <= 43:  return "Leather and hides"
    elif 44 <= hs2 <= 49:  return "Wood, paper"
    elif 50 <= hs2 <= 63:  return "Textiles"
    elif 64 <= hs2 <= 67:  return "Footwear / headgear"
    elif 68 <= hs2 <= 70:  return "Stone, cement, glass"
    elif 71 <= hs2 <= 71:  return "Precious metals / stones"
    elif 72 <= hs2 <= 83:  return "Base metals"
    elif 84 <= hs2 <= 85:  return "Machinery / electrical"
    elif 86 <= hs2 <= 89:  return "Transport equipment"
    elif 90 <= hs2 <= 92:  return "Instruments"
    elif 93 <= hs2 <= 93:  return "Arms / ammunition"
    elif 94 <= hs2 <= 96:  return "Miscellaneous manufactures"
    elif hs2 == 97:        return "Art / antiques"
    else:                  return "Other"

df["sector"] = df["hs2"].apply(hs_section).astype("category")

print("Sector distribution:")
print(df["sector"].value_counts())

In [ ]:
# Algeria exports, partner demand, opportunity rows, and average penetration
sector_summary_base = (
    df.groupby("sector", as_index=False, observed=True)
    .agg(
        alg_exports=("alg_export_v", "sum"),
        partner_demand=("partner_import_v", "sum"),
        opportunity_rows=("label_opportunity", "sum"),
        avg_penetration=("market_penetration", "mean")
    )
)

# Correct world demand aggregation:
# world_import_v is repeated for every partner, so keep only one row per (year, product)
world_sector = (
    df[["t", "k", "sector", "world_import_v"]]
    .drop_duplicates(subset=["t", "k"])
    .groupby("sector", as_index=False, observed=True)["world_import_v"]
    .sum()
    .rename(columns={"world_import_v": "world_demand"})
)

sector_summary = sector_summary_base.merge(world_sector, on="sector", how="left")

# Convert BACI values from thousand USD to USD
sector_summary["alg_exports_usd"] = sector_summary["alg_exports"] * 1_000
sector_summary["partner_demand_usd"] = sector_summary["partner_demand"] * 1_000
sector_summary["world_demand_usd"] = sector_summary["world_demand"] * 1_000

sector_summary = sector_summary.sort_values("opportunity_rows", ascending=False)

sector_summary[[
    "sector",
    "alg_exports_usd",
    "world_demand_usd",
    "opportunity_rows",
    "avg_penetration"
]]

The average market penetration values are very close to zero in most sectors, which means Algeria supplies only a very small share of global partner demand in many product categories.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Plot 1: Opportunity rows by sector
axes[0].barh(
    sector_summary["sector"][::-1],
    sector_summary["opportunity_rows"][::-1],
    color="#e67e22"
)
axes[0].set_title("Opportunity Rows by Sector", fontsize=12)
axes[0].set_xlabel("Number of opportunity rows")
axes[0].grid(True, axis="x", linestyle="--", alpha=0.5)

# Plot 2: Algeria actual exports by sector
top_sectors_exp = sector_summary.sort_values("alg_exports_usd", ascending=False)

axes[1].barh(
    top_sectors_exp["sector"][::-1],
    top_sectors_exp["alg_exports_usd"][::-1] / 1e9,
    color="#27ae60"
)
axes[1].set_title("Algeria Export Value by Sector (2012–2023)", fontsize=12)
axes[1].set_xlabel("Export Value (billion USD)")
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.0f}B"))
axes[1].grid(True, axis="x", linestyle="--", alpha=0.5)

plt.suptitle("Sector-Level Trade Analysis", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

The sector-level analysis shows a strong contrast between Algeria's current export structure and potential opportunity areas.

Algeria's actual exports are highly concentrated in the `Mineral products` sector, which includes petroleum oils and gas products. This confirms the previous finding that Algeria's export basket is strongly dominated by hydrocarbons.

However, the largest number of opportunity rows appears in sectors such as chemicals, textiles, machinery/electrical products, base metals, and animal products. These sectors contain many country-product pairs where Algeria has low market penetration while partner demand is relatively high.

This is important for the project because it shows that export opportunities are not necessarily located in the sectors where Algeria currently exports the most. The model should therefore help identify promising products and markets outside the current dominant mineral export structure.

For global demand, we avoided double-counting by keeping only one row per `(year, product)` before aggregating by sector, because `world_import_v` is repeated across partner rows in the full opportunity grid.

---
## Section 12 — EDA: Opportunity Countries and Products

We now focus on the opportunity subset (label = 1) to answer two key strategic questions:
1. **Which countries** have the most potential products for Algeria to export to?
2. **Which products** have the most potential destination markets?

In [ ]:
# Opportunity subset
opp = df[df["label_opportunity"] == 1].copy()

print(f"Total opportunity rows: {len(opp):,}")
print(f"Unique opportunity partners: {opp['j'].nunique()}")
print(f"Unique opportunity products: {opp['k'].nunique()}")

The opportunity subset contains 3,007,224 rows where `label_opportunity = 1`.

All 227 partner countries and all 5,196 active HS6 products appear at least once in the opportunity subset. This is expected because the label is defined at the year-product-partner level, so a product or country may be an opportunity in some years or markets but not necessarily in all cases.

In [ ]:
# Build small partner lookup table
partner_lookup = (
    pd.read_parquet(
        MASTER_PATH,
        columns=["j", "country_name"],
        engine="fastparquet"
    )
    .dropna()
    .drop_duplicates()
)

# Fix encoding issue for Türkiye if present
partner_lookup["country_name"] = (
    partner_lookup["country_name"]
    .astype(str)
    .str.replace("TÃ¼rkiye", "Türkiye", regex=False)
    .str.replace("TÃƒÂ¼rkiye", "Türkiye", regex=False)
)

# Top countries by number of opportunity products
opp_by_country = (
    opp.groupby("j", as_index=False, observed=True)
    .agg(
        unique_products=("k", "nunique"),
        opportunity_rows=("k", "size"),
        total_partner_import_v=("partner_import_v", "sum"),
        avg_market_penetration=("market_penetration", "mean")
    )
    .merge(partner_lookup, on="j", how="left")
)

opp_by_country["total_partner_import_usd"] = (
    opp_by_country["total_partner_import_v"] * 1_000
)

opp_by_country = (
    opp_by_country
    .sort_values("unique_products", ascending=False)
    .head(20)
)

fig, ax = plt.subplots(figsize=(10, 7))

ax.barh(
    opp_by_country["country_name"][::-1],
    opp_by_country["unique_products"][::-1],
    color="#c0392b"
)

ax.set_title("Top 20 Partner Countries by Number of Opportunity HS6 Products", fontsize=13)
ax.set_xlabel("Number of opportunity products")
ax.set_ylabel("Country")
ax.grid(True, axis="x", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

opp_by_country[[
    "country_name",
    "unique_products",
    "opportunity_rows",
    "total_partner_import_usd",
    "avg_market_penetration"
]]

The opportunity country ranking shows that large importing economies contain the highest number of potential products for Algeria.

The USA, United Kingdom, Germany, Netherlands, Spain, and Italy appear at the top. These countries import a very wide range of HS6 products, while Algeria's market penetration remains very low in many of those product-country combinations.

This does not mean that all products are equally realistic export opportunities. It means that these countries contain many product categories where demand exists and Algeria is currently underrepresented.

The average market penetration values are displayed as 0.00 because they are extremely small after rounding. This confirms that Algeria supplies only a very small share of these partner countries' imports for the selected opportunity rows.

This result is useful for strategy: large and diversified import markets may offer many opportunities, but later modeling and ranking should prioritize them using additional criteria such as demand size, RCA, distance, GDP, and product feasibility.

In [ ]:
# Top products by number of opportunity countries
# The lightweight df does not contain description_short,
# so we rebuild a small lookup table from the master parquet.

product_lookup = (
    pd.read_parquet(
        MASTER_PATH,
        columns=["k", "description_short"],
        engine="fastparquet"
    )
    .dropna()
    .drop_duplicates()
)

opp_by_product = (
    opp.groupby("k", as_index=False)
    .agg(
        opportunity_countries  = ("j", "nunique"),
        opportunity_rows       = ("j", "size"),
        total_partner_import_v = ("partner_import_v", "sum"),
        avg_rca                = ("rca", "mean"),
        avg_market_penetration = ("market_penetration", "mean")
    )
    .merge(product_lookup, on="k", how="left")
    .sort_values(["opportunity_countries", "total_partner_import_v"], ascending=[False, False])
    .head(20)
)

opp_by_product["total_partner_import_usd"] = opp_by_product["total_partner_import_v"] * 1_000

fig, ax = plt.subplots(figsize=(11, 7))
ax.barh(
    opp_by_product["description_short"][::-1],
    opp_by_product["opportunity_countries"][::-1],
    color="#2980b9"
)
ax.set_title("Top 20 Products by Number of Opportunity Countries", fontsize=13)
ax.set_xlabel("Number of potential destination countries")
ax.set_ylabel("HS6 product")
ax.grid(True, axis="x", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

opp_by_product[
    ["k", "description_short", "opportunity_countries",
     "opportunity_rows", "total_partner_import_usd",
     "avg_rca", "avg_market_penetration"]
]

The opportunity product analysis shows which HS6 products appear as potential opportunities across the largest number of partner countries.

Several products have opportunities in all 227 partner countries. This means that, according to the label definition, Algeria has very low market penetration for these products while partner import demand exists across many markets.

However, because many products have the same number of opportunity countries, the bar chart alone is not enough to compare them. The table is important because it also shows total partner import demand, RCA, and average market penetration.

Some products have missing average RCA values. This usually means Algeria has little or no recorded export activity for those products, so RCA could not be calculated. This does not necessarily remove them from consideration, but it means they need further feasibility analysis.

Overall, this section identifies products with broad international opportunity coverage. Later ranking should combine number of opportunity countries with partner demand, RCA, distance, and market feasibility.

In [ ]:
# Better comparison: among broad-opportunity products, rank by total partner demand
opp_by_product_demand = opp_by_product.sort_values(
    "total_partner_import_usd",
    ascending=False
)

fig, ax = plt.subplots(figsize=(11, 7))

ax.barh(
    opp_by_product_demand["description_short"][::-1],
    opp_by_product_demand["total_partner_import_usd"][::-1] / 1e9,
    color="#34495e"
)

ax.set_title("Top Opportunity Products by Total Partner Import Demand", fontsize=13)
ax.set_xlabel("Total partner import demand (billion USD)")
ax.set_ylabel("HS6 product")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.0f}B"))
ax.grid(True, axis="x", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

This second chart improves the product opportunity analysis by ranking the same broad-opportunity products according to total partner import demand.

In the previous chart, many products had the same number of opportunity countries, so the bars were almost identical. Here, demand size helps separate the products more clearly.

Products such as ceramic flags and pavings, spacecraft/satellites, coats, vessels, and machine-tools show large partner import demand while Algeria has very low market penetration. These products may represent interesting opportunity areas, but they should not be selected automatically. They still need to be evaluated using additional criteria such as Algeria’s production capacity, RCA, logistics, distance, and feasibility.

This confirms why the final opportunity ranking should combine several variables, not only the number of opportunity countries.

---
## Section 13 — EDA: Country-Product Demand Heatmap

The heatmap shows the **partner import value** for top opportunity country-product pairs. Darker cells indicate higher demand — these are the most attractive combinations for Algerian exporters.

We use a log scale to handle the large range of import values and keep the heatmap readable.

In [ ]:
# Actionable opportunity subset
# We keep the original opportunity label, but for EDA visualizations
# we remove very small-demand cases.
# BACI values are in thousand USD, so 1,000 = 1 million USD.

MIN_PARTNER_DEMAND_KUSD = 1_000

opp_actionable = opp[opp["partner_import_v"] >= MIN_PARTNER_DEMAND_KUSD].copy()

print(f"Original opportunity rows:   {len(opp):,}")
print(f"Actionable opportunity rows: {len(opp_actionable):,}")
print(f"Minimum partner demand:      ${MIN_PARTNER_DEMAND_KUSD:,}k = ${MIN_PARTNER_DEMAND_KUSD*1000:,}")
print(f"Unique partners:             {opp_actionable['j'].nunique()}")
print(f"Unique products:             {opp_actionable['k'].nunique()}")

# Small partner lookup
partner_lookup = (
    pd.read_parquet(
        MASTER_PATH,
        columns=["j", "country_name"],
        engine="fastparquet"
    )
    .dropna()
    .drop_duplicates()
)

# Fix Türkiye encoding if needed
partner_lookup["country_name"] = (
    partner_lookup["country_name"]
    .astype(str)
    .str.replace("TÃ¼rkiye", "Türkiye", regex=False)
    .str.replace("TÃƒÂ¼rkiye", "Türkiye", regex=False)
)

# Small product lookup
product_lookup = (
    pd.read_parquet(
        MASTER_PATH,
        columns=["k", "description_short"],
        engine="fastparquet"
    )
    .dropna()
    .drop_duplicates()
)

In [ ]:
# Select top 15 countries and top 15 products from actionable opportunity rows
top_c = (
    opp_actionable.groupby("j", observed=True)
    .size()
    .sort_values(ascending=False)
    .head(15)
    .index
)

top_p = (
    opp_actionable.groupby("k", observed=True)
    .size()
    .sort_values(ascending=False)
    .head(15)
    .index
)

# Build heatmap dataframe
heatmap_base = (
    opp_actionable.loc[
        opp_actionable["j"].isin(top_c) & opp_actionable["k"].isin(top_p),
        ["t", "j", "k", "alg_export_v", "partner_import_v", "market_penetration", "label_opportunity"]
    ]
    .merge(partner_lookup, on="j", how="left")
    .merge(product_lookup, on="k", how="left")
)

# Partner import demand heatmap
pivot_demand = heatmap_base.pivot_table(
    index="country_name",
    columns="description_short",
    values="partner_import_v",
    aggfunc="sum",
    fill_value=0
)

pivot_log = np.log1p(pivot_demand)

fig, ax = plt.subplots(figsize=(16, 7))

im = ax.imshow(
    pivot_log.values,
    aspect="auto",
    cmap="YlOrRd"
)

plt.colorbar(im, ax=ax, label="log(1 + partner import value in thousand USD)")

ax.set_xticks(range(len(pivot_log.columns)))
ax.set_xticklabels(pivot_log.columns, rotation=45, ha="right", fontsize=8)

ax.set_yticks(range(len(pivot_log.index)))
ax.set_yticklabels(pivot_log.index, fontsize=9)

ax.set_title(
    "Country × Product Heatmap — Actionable Opportunity Partner Import Demand",
    fontsize=12
)
ax.set_xlabel("Product")
ax.set_ylabel("Country")

plt.tight_layout()
plt.show()

pivot_demand.head()

In [ ]:
# Diagnostic: Algeria's current presence in selected actionable opportunity pairs

print("Verification of selected actionable heatmap opportunity pairs")
print(f"Rows: {len(heatmap_base):,}")
print(f"Years: {heatmap_base['t'].nunique()}")
print(f"Countries: {heatmap_base['j'].nunique()}")
print(f"Products: {heatmap_base['k'].nunique()}")

print("\nAlgeria export value summary:")
print(heatmap_base["alg_export_v"].describe())

print("\nPartner import value summary:")
print(heatmap_base["partner_import_v"].describe())

print("\nMarket penetration summary:")
print(heatmap_base["market_penetration"].describe())

print("\nAll rows are opportunity rows:")
print((heatmap_base["label_opportunity"] == 1).all())

if heatmap_base["market_penetration"].max() == 0:
    print("\nAll selected actionable opportunity pairs have zero Algerian market penetration.")
    print("This means Algeria has no recorded exports in these selected high-demand country-product pairs.")
else:
    print("\nSome selected opportunity pairs have small but non-zero Algerian market penetration.")

The actionable heatmap focuses only on opportunity rows where partner import demand is at least $1 million.

The verification confirms that the selected heatmap subset contains 15 countries and 15 products across 12 years, but not all 2,700 possible combinations remain because some low-demand rows were removed by the actionable filter.

All selected rows are valid opportunity rows, and Algeria's export value is zero for all of them. This means Algeria has no recorded exports in these selected high-demand country-product pairs.

At the same time, partner import demand is positive and above the minimum-demand threshold. Therefore, the heatmap highlights markets where demand exists, but Algeria is currently absent.

This makes the heatmap more useful for export strategy because it avoids tiny-demand cases and focuses on more realistic opportunity combinations.

---
## Section 14 — Prepare Datasets for Modeling

We save ready-to-use train/validation/test splits as Parquet files for the next notebooks (clustering, classification, forecasting).
The scaled arrays are saved as numpy files for models that require them.

In [ ]:
import pyarrow.parquet as pq
import gc

modeling_dir = DATA_DIR / "modeling"
modeling_dir.mkdir(exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════
# PHASE 1 — Save complete splits WITHOUT loading the full master into RAM
# ═══════════════════════════════════════════════════════════════════════════
# PyArrow's filter pushdown reads only matching rows from disk.
# Peak RAM = one split at a time (~3–5M rows × 50 cols) instead of 14M × 50.

print("── Phase 1: saving full-column splits (memory-safe) ──")

available_cols = None  # captured from first split

for split_name in ["train", "val", "test"]:
    # Read only rows where split == split_name — never loads the full master
    table = pq.read_table(
        MASTER_PATH,
        filters=[("split", "==", split_name)]
    )
    split_df = table.to_pandas()
    del table  # free arrow buffer immediately

    yrs = f"{int(split_df['t'].min())}–{int(split_df['t'].max())}"
    print(f"  {split_name:5s}: {len(split_df):,} rows × {split_df.shape[1]} cols  ({yrs})")

    split_df.to_parquet(
        modeling_dir / f"{split_name}.parquet",
        index=False, engine="fastparquet"
    )

    if available_cols is None:
        available_cols = set(split_df.columns)

    del split_df
    gc.collect()

# ── Confirm scale columns (check missingness from saved train split) ────────
# We load ONLY the scale candidates from train.parquet to check NaN rates,
# avoiding any need to reload the full master.
scale_check = pd.read_parquet(
    modeling_dir / "train.parquet",
    columns=[c for c in SCALE_FEATURES if c in available_cols],
    engine="fastparquet"
)
scale_cols = [
    c for c in SCALE_FEATURES
    if c in available_cols and scale_check[c].isna().mean() < 1.0
]
del scale_check
gc.collect()

print(f"\n  Scale features available: {len(scale_cols)} / {len(SCALE_FEATURES)}")
if len(scale_cols) < len(SCALE_FEATURES):
    print(f"  Dropped: {sorted(set(SCALE_FEATURES) - set(scale_cols))}")
print("  master_full never loaded — memory-safe.")

# ═══════════════════════════════════════════════════════════════════════════
# PHASE 2 — Memory-safe scaling
# ═══════════════════════════════════════════════════════════════════════════
# sklearn's SimpleImputer allocates a boolean NaN-mask of shape
# (n_rows × n_cols) before imputing.  On 9.4M × 31 that is ~279 MiB on top
# of the data itself — enough to crash on a typical laptop.
#
# We avoid this by:
#   1. Loading only the scale columns from the saved parquets (not all 50).
#   2. Computing imputation medians column-by-column (1-D, no large mask).
#   3. Applying mean/std standardisation column-by-column in float32.
#   4. Saving the fit parameters to JSON so any notebook can re-apply them.

print("\n── Phase 2: memory-safe fit & scale ──")

# Load only scale columns + label from each saved split
load_cols = scale_cols + ["label_opportunity"]

train_sc = pd.read_parquet(modeling_dir / "train.parquet", columns=load_cols, engine="fastparquet")
val_sc   = pd.read_parquet(modeling_dir / "val.parquet",   columns=load_cols, engine="fastparquet")
test_sc  = pd.read_parquet(modeling_dir / "test.parquet",  columns=load_cols, engine="fastparquet")

y_train = train_sc.pop("label_opportunity").values.astype(np.int8)
y_val   = val_sc.pop("label_opportunity").values.astype(np.int8)
y_test  = test_sc.pop("label_opportunity").values.astype(np.int8)

# Fit: compute per-column median (for NaN fill) and post-fill mean/std
# All operations are 1-D — no large intermediate arrays.
fit_params = {}  # saved to JSON for use in modeling notebooks

for col in scale_cols:
    vals = train_sc[col].values.astype(np.float32)
    median = float(np.nanmedian(vals))
    vals_filled = np.where(np.isnan(vals), median, vals)
    mean = float(vals_filled.mean())
    std  = float(vals_filled.std())
    std  = std if std > 0 else 1.0   # avoid divide-by-zero on constant columns
    fit_params[col] = {"median": median, "mean": mean, "std": std}

# Save fit parameters
(modeling_dir / "scale_params.json").write_text(
    _json.dumps({"features": scale_cols, "params": fit_params}, indent=2)
)
print("  Fit parameters computed and saved to scale_params.json")

# Transform: fill NaNs then standardise, column-by-column, output float32
def apply_scaling(df, fit_params, scale_cols):
    """Apply saved median imputation + z-score standardisation column-by-column.
    Returns a float32 numpy array.  Peak extra memory = 1 extra column at a time.
    """
    out = np.empty((len(df), len(scale_cols)), dtype=np.float32)
    for i, col in enumerate(scale_cols):
        p = fit_params[col]
        vals = df[col].values.astype(np.float32)
        vals = np.where(np.isnan(vals), p["median"], vals)
        out[:, i] = (vals - p["mean"]) / p["std"]
    return out

print("  Scaling train split...", end=" ", flush=True)
X_train_scaled = apply_scaling(train_sc, fit_params, scale_cols)
print(f"{X_train_scaled.shape}")

print("  Scaling val   split...", end=" ", flush=True)
X_val_scaled   = apply_scaling(val_sc,   fit_params, scale_cols)
print(f"{X_val_scaled.shape}")

print("  Scaling test  split...", end=" ", flush=True)
X_test_scaled  = apply_scaling(test_sc,  fit_params, scale_cols)
print(f"{X_test_scaled.shape}")

del train_sc, val_sc, test_sc
gc.collect()

# Save scaled arrays and labels
np.save(modeling_dir / "X_train_scaled.npy", X_train_scaled)
np.save(modeling_dir / "X_val_scaled.npy",   X_val_scaled)
np.save(modeling_dir / "X_test_scaled.npy",  X_test_scaled)
np.save(modeling_dir / "y_train.npy",        y_train)
np.save(modeling_dir / "y_val.npy",          y_val)
np.save(modeling_dir / "y_test.npy",         y_test)

del X_train_scaled, X_val_scaled, X_test_scaled
gc.collect()

print("\n── Summary ──")
print("  Saved to data/modeling/:")
for fname in sorted(modeling_dir.iterdir()):
    size_mb = fname.stat().st_size / 1e6
    print(f"    {fname.name:<35s} {size_mb:>7.1f} MB")
print("\n  Fit parameters computed on train split ONLY — no data leakage.")


The modeling datasets were saved successfully using a two-phase approach that avoids the `MemoryError`.

**Phase 1** loads the full 50-column master parquet, writes each split to its own Parquet file, then immediately frees `master_full` from memory before any scaling begins. Each of `train.parquet`, `val.parquet`, and `test.parquet` contains every column, so downstream notebooks can load exactly what they need.

**Phase 2** reloads only the `scale_cols` (31 columns) from the already-saved parquets — far less memory than keeping `master_full` alive. Imputation and scaling are done **column-by-column** rather than via `sklearn.Pipeline.fit_transform`, which avoids allocating a boolean NaN-mask of shape `(9.4M × 31)` all at once. The output is stored as `float32` (half the size of sklearn's default `float64`).

The fit parameters (per-column median, mean, std) are saved to `scale_params.json`. Any modeling notebook can reproduce or extend this scaling without rerunning this cell:

```python
import json
params = json.load(open("data/modeling/scale_params.json"))
# then apply params["params"][col]["median"] / ["mean"] / ["std"] as needed
```

Labels are saved as `int8` (0/1), halving their footprint versus the default `int64`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path("..") if Path.cwd().name == "notebooks" else Path(".")
MODEL_DIR = ROOT / "data" / "modeling"

files_to_check = [
    "train.parquet", "val.parquet", "test.parquet",
    "X_train_scaled.npy", "X_val_scaled.npy", "X_test_scaled.npy",
    "y_train.npy", "y_val.npy", "y_test.npy"
]

print("Saved file check:")
for f in files_to_check:
    path = MODEL_DIR / f
    print(f"  {'✓' if path.exists() else '✗'} {path}")

print("\nArray shapes:")
print("  X_train:", np.load(MODEL_DIR / "X_train_scaled.npy").shape)
print("  X_val:  ", np.load(MODEL_DIR / "X_val_scaled.npy").shape)
print("  X_test: ", np.load(MODEL_DIR / "X_test_scaled.npy").shape)
print("  y_train:", np.load(MODEL_DIR / "y_train.npy").shape)
print("  y_val:  ", np.load(MODEL_DIR / "y_val.npy").shape)
print("  y_test: ", np.load(MODEL_DIR / "y_test.npy").shape)

---
## Section 15 — Save EDA Outputs for Dashboard

We export the key summary tables computed during EDA as CSV files. These will serve as the data layer for the Grafana / Metabase dashboard required by the project.

In [ ]:
# Save EDA summary outputs for dashboard use

eda_dir = DATA_DIR / "eda_outputs"
eda_dir.mkdir(parents=True, exist_ok=True)

# Remove old CSV files first to avoid mixing old outputs with updated ones
for old_file in eda_dir.glob("*.csv"):
    old_file.unlink()

# Save main EDA summary tables
exports_by_year.to_csv(eda_dir / "exports_by_year.csv", index=False)
top_partners.to_csv(eda_dir / "top_export_partners.csv", index=False)
top_products.to_csv(eda_dir / "top_export_products.csv", index=False)
global_demand_products.to_csv(eda_dir / "global_demand_products.csv", index=False)
opp_by_country.to_csv(eda_dir / "opportunities_by_country.csv", index=False)
opp_by_product.to_csv(eda_dir / "opportunities_by_product.csv", index=False)
sector_summary.to_csv(eda_dir / "sector_summary.csv", index=False)

# Save country-product heatmap tables
pivot_demand.to_csv(eda_dir / "country_product_heatmap_demand.csv")
pivot_log.to_csv(eda_dir / "country_product_heatmap_log.csv")

print("EDA outputs saved to data/eda_outputs/:")
for f in sorted(eda_dir.iterdir()):
    print(f"  {f.name}")

The EDA summary tables were exported successfully as CSV files.

These files contain the main aggregated results used during the exploratory analysis: yearly exports, top export partners, top exported products, global demand products, opportunity countries, opportunity products, sector summaries, and country-product heatmap tables.

Both the raw demand heatmap and the log-transformed heatmap were saved. The raw demand table is useful for exact values, while the log version is useful for dashboard visualizations because it reduces the effect of very large import values.

Saving these outputs is useful because the dashboard does not need to reload the full 14 million-row dataset. Instead, Grafana, Metabase, or any dashboarding tool can directly use these smaller CSV summary files.

This step creates the data layer for the visualization/dashboard part of the project.

---

## Notebook 02 Summary

This notebook completed the **Data Cleaning, Exploratory Data Analysis, and Modeling Preparation** stage of the project.

| Task | Status |
|------|--------|
| Data source audit | ✓ Done |
| Duplicate check | ✓ No duplicates |
| Infinite value check | ✓ None found |
| Missing value analysis | ✓ Documented with imputation plan |
| Normalization / scaling plan | ✓ Done — fitted on train only |
| EDA: export flows over time | ✓ Done |
| EDA: top export partners | ✓ Done |
| EDA: top exported products | ✓ Done |
| EDA: global demand analysis | ✓ Done |
| EDA: sector demand analysis | ✓ Done |
| EDA: opportunity countries | ✓ Done |
| EDA: opportunity products | ✓ Done |
| EDA: country-product heatmaps | ✓ Done |
| Modeling-ready train / validation / test splits saved | ✓ Done |
| Dashboard CSV exports saved | ✓ Done |

### Key findings

Algeria’s exports are strongly concentrated in **mineral products**, especially petroleum oils and gas-related products. This confirms the need for export diversification.

The opportunity analysis shows that many potential markets exist outside Algeria’s current dominant export structure. Sectors such as **chemicals**, **textiles**, **machinery/electrical products**, **base metals**, and **animal products** contain many opportunity rows.

The country-product heatmap highlights actionable opportunity pairs where partner demand exists but Algeria has no recorded exports. This supports the main project objective: identifying underexploited global export opportunities for Algerian exporters.

### Outputs created

This notebook saved:

- modeling-ready train, validation, and test datasets,
- scaled NumPy arrays for models that require normalization,
- EDA summary CSV files for dashboarding.

**Next notebooks:** `03_clustering.ipynb`, `04_classification.ipynb`, `05_forecasting.ipynb`